# PyTorch 기반 DNN-GAN 손글씨 이미지 생성 실습

이 노트북은 **PyTorch만 사용하여 DNN 기반 GAN 모델을 구현**하는 실습 파일입니다.

GAN은 두 개의 신경망을 함께 학습시키는 생성 모델입니다.  
**생성자(Generator)** 는 무작위 노이즈를 입력받아 가짜 이미지를 만들고, **감별자(Discriminator)** 는 입력 이미지가 실제 이미지인지 생성 이미지인지 판별합니다.

이 예제에서는 MNIST 손글씨 숫자 이미지를 사용하여 28×28 크기의 흑백 숫자 이미지를 생성합니다.

## 1. 라이브러리 불러오기

학습에 필요한 PyTorch, torchvision, matplotlib, numpy, 파일 처리 라이브러리를 불러옵니다.

In [ ]:
# os 모듈은 폴더 생성, 경로 결합, 파일 삭제 같은 운영체제 관련 작업을 처리할 때 사용합니다.
import os

# glob 모듈은 특정 확장자나 이름 패턴에 맞는 파일 목록을 찾을 때 사용합니다.
import glob

# time 함수는 학습 시작 시각과 종료 시각을 기록하여 전체 학습 시간을 계산할 때 사용합니다.
from time import time

# numpy는 배열 계산과 난수 처리를 위해 사용하는 대표적인 수치 계산 라이브러리입니다.
import numpy as np

# matplotlib.pyplot은 이미지나 그래프를 화면에 출력하고 파일로 저장할 때 사용합니다.
import matplotlib.pyplot as plt

# torch는 PyTorch의 핵심 라이브러리로 텐서 연산, GPU 연산, 자동미분 기능을 제공합니다.
import torch

# torch.nn은 신경망 계층, 활성화 함수, 손실 함수 등을 만들 때 사용하는 모듈입니다.
import torch.nn as nn

# torch.optim은 Adam, SGD 같은 최적화 알고리즘을 제공하는 모듈입니다.
import torch.optim as optim

# DataLoader는 데이터셋을 미니배치 단위로 나누어 모델 학습에 공급하는 도구입니다.
from torch.utils.data import DataLoader

# torchvision.datasets는 MNIST 같은 공개 이미지 데이터셋을 쉽게 불러오는 기능을 제공합니다.
from torchvision import datasets

# torchvision.transforms는 이미지 데이터를 텐서로 변환하거나 정규화하는 전처리 기능을 제공합니다.
from torchvision import transforms

# GPU가 사용 가능하면 cuda 장치를 사용하고, 그렇지 않으면 CPU 장치를 사용합니다.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 현재 학습에 사용될 장치를 출력하여 GPU 사용 여부를 확인합니다.
print("사용 장치:", device)

## 2. 하이퍼파라미터 설정

하이퍼파라미터는 모델 구조와 학습 방식을 결정하는 값입니다.  
GAN은 학습이 불안정할 수 있으므로 학습률, 배치 크기, 은닉층 크기 등을 명확하게 관리하는 것이 중요합니다.

In [ ]:
# 생성자 은닉층의 뉴런 수입니다. 값이 클수록 생성자가 표현할 수 있는 패턴이 많아집니다.
M_GEN = 128

# 감별자 은닉층의 뉴런 수입니다. 값이 클수록 진짜 이미지와 가짜 이미지를 구분하는 능력이 커질 수 있습니다.
M_DIS = 128

# 생성자에 입력되는 무작위 노이즈 벡터의 차원 수입니다.
M_NOISE = 100

# MNIST 이미지는 흑백 1채널, 높이 28, 너비 28 구조입니다.
M_SHAPE = (1, 28, 28)

# 전체 학습 반복 횟수입니다. 빠른 테스트를 원하면 500 정도로 줄여 실행할 수 있습니다.
M_EPOCH = 5000

# 한 번의 학습 단계에서 사용할 이미지 개수입니다.
M_BATCH = 300

# 학습률은 가중치를 한 번 업데이트할 때 얼마나 크게 이동할지 정하는 값입니다.
M_LR = 0.0002

# Adam 최적화 알고리즘에서 1차 모멘텀 이동평균에 사용하는 값입니다.
M_BETA1 = 0.5

# 학습 중 생성 이미지가 저장될 폴더 이름입니다.
M_FOLDER = "output"

# output 폴더가 없으면 생성하고, 이미 있으면 그대로 사용합니다.
os.makedirs(M_FOLDER, exist_ok=True)

# 기존 output 폴더 안의 png 이미지를 찾아 새 학습 결과와 섞이지 않도록 삭제합니다.
for file_path in glob.glob(os.path.join(M_FOLDER, "*.png")):
    # 찾은 이미지 파일을 하나씩 삭제합니다.
    os.remove(file_path)

# 실험 결과를 최대한 재현할 수 있도록 CPU 난수 시드를 고정합니다.
torch.manual_seed(42)

# GPU가 사용 가능한 경우 GPU 난수 시드도 함께 고정합니다.
if torch.cuda.is_available():
    # 현재 사용하는 GPU의 난수 시드를 고정합니다.
    torch.cuda.manual_seed(42)

# 설정한 주요 하이퍼파라미터를 출력하여 확인합니다.
print("노이즈 차원:", M_NOISE)
print("배치 크기:", M_BATCH)
print("학습 반복 횟수:", M_EPOCH)
print("학습률:", M_LR)

## 3. MNIST 데이터셋 준비

GAN은 숫자 정답 라벨을 직접 사용하지 않습니다.  
여기서는 MNIST 이미지 데이터만 사용하여 생성 모델을 학습합니다.

이미지 픽셀값은 원래 0~1 범위로 변환되지만, 생성자의 마지막 활성화 함수 출력 범위에 맞추기 위해 `[-1, 1]` 범위로 정규화합니다.

In [ ]:
# Compose는 여러 전처리 단계를 순서대로 묶어 하나의 전처리 파이프라인으로 만들어 줍니다.
transform = transforms.Compose([
    # ToTensor는 이미지를 PyTorch 텐서로 변환하고 픽셀값을 0~1 범위로 바꿉니다.
    transforms.ToTensor(),

    # Normalize는 픽셀값을 평균 0.5, 표준편차 0.5 기준으로 변환하여 대략 -1~1 범위로 만듭니다.
    transforms.Normalize((0.5,), (0.5,))
])

# MNIST 학습 데이터셋을 준비합니다.
train_dataset = datasets.MNIST(
    # 데이터가 저장될 기본 폴더입니다.
    root="./data",

    # train=True는 학습용 데이터 60,000장을 사용하겠다는 뜻입니다.
    train=True,

    # 데이터가 없으면 자동으로 다운로드합니다.
    download=True,

    # 위에서 정의한 전처리를 각 이미지에 적용합니다.
    transform=transform
)

# DataLoader는 데이터셋을 학습에 사용할 수 있도록 미니배치 단위로 묶어 줍니다.
train_loader = DataLoader(
    # 사용할 데이터셋을 지정합니다.
    train_dataset,

    # 한 번에 모델에 입력할 이미지 개수를 지정합니다.
    batch_size=M_BATCH,

    # 매 반복마다 데이터 순서를 섞어 모델이 특정 순서에 과하게 적응하지 않게 합니다.
    shuffle=True,

    # 마지막 배치 크기가 M_BATCH보다 작으면 버려서 항상 같은 배치 크기를 유지합니다.
    drop_last=True
)

# 전체 학습 이미지 개수를 출력합니다.
print("MNIST 학습 이미지 개수:", len(train_dataset))

# DataLoader에서 첫 번째 배치를 꺼내 이미지 텐서 구조를 확인합니다.
sample_images, sample_labels = next(iter(train_loader))

# 이미지 배치의 차원을 출력합니다. 예상 형태는 [배치크기, 채널, 높이, 너비]입니다.
print("이미지 배치 형태:", sample_images.shape)

## 4. 생성자 모델 정의

생성자는 100차원 무작위 노이즈를 입력받아 28×28 흑백 이미지를 출력합니다.  
완전연결층을 여러 개 쌓은 DNN 구조로 구성합니다.

In [ ]:
# Generator 클래스는 무작위 노이즈를 입력받아 가짜 이미지를 생성하는 신경망입니다.
class Generator(nn.Module):
    # __init__ 메서드는 모델에 들어갈 계층을 정의하는 초기화 함수입니다.
    def __init__(self):
        # 부모 클래스 nn.Module의 초기화 기능을 먼저 실행합니다.
        super().__init__()

        # Sequential은 여러 계층을 순서대로 연결하여 하나의 모델처럼 사용할 수 있게 합니다.
        self.model = nn.Sequential(
            # 첫 번째 완전연결층입니다. 100차원 노이즈를 128차원 특징으로 변환합니다.
            nn.Linear(M_NOISE, M_GEN),

            # LeakyReLU는 음수 입력도 작은 값으로 통과시켜 기울기 소실 문제를 줄입니다.
            nn.LeakyReLU(0.01),

            # 두 번째 완전연결층입니다. 128차원 특징을 다시 128차원 특징으로 변환합니다.
            nn.Linear(M_GEN, M_GEN),

            # 두 번째 은닉층에도 비선형 활성화 함수를 적용하여 복잡한 패턴을 학습하게 합니다.
            nn.LeakyReLU(0.01),

            # 출력층입니다. 128차원 특징을 28*28=784개의 픽셀값으로 변환합니다.
            nn.Linear(M_GEN, 28 * 28),

            # Tanh는 출력값을 -1부터 1 사이로 제한하여 정규화된 이미지 범위와 맞춥니다.
            nn.Tanh()
        )

    # forward 메서드는 입력 데이터가 모델 내부에서 계산되는 흐름을 정의합니다.
    def forward(self, z):
        # z는 [배치크기, 노이즈차원] 형태의 무작위 노이즈 텐서입니다.
        out = self.model(z)

        # 784차원 벡터를 [배치크기, 1, 28, 28] 이미지 형태로 변환합니다.
        img = out.view(z.size(0), 1, 28, 28)

        # 생성된 이미지 텐서를 반환합니다.
        return img

# 생성자 모델 객체를 생성합니다.
generator = Generator()

# 생성자 모델을 GPU 또는 CPU 장치로 이동합니다.
generator = generator.to(device)

# 생성자 구조를 출력하여 계층 구성을 확인합니다.
print(generator)

## 5. 감별자 모델 정의

감별자는 입력 이미지가 실제 MNIST 이미지인지, 생성자가 만든 이미지인지 판별합니다.  
출력값은 0~1 사이의 확률값이며, 1에 가까울수록 실제 이미지라고 판단합니다.

In [ ]:
# Discriminator 클래스는 이미지가 실제 이미지인지 생성 이미지인지 판별하는 신경망입니다.
class Discriminator(nn.Module):
    # __init__ 메서드는 감별자 내부 계층을 정의합니다.
    def __init__(self):
        # 부모 클래스 nn.Module의 초기화 기능을 실행합니다.
        super().__init__()

        # 감별자 모델을 순차 구조로 정의합니다.
        self.model = nn.Sequential(
            # Flatten은 [배치크기, 1, 28, 28] 이미지를 [배치크기, 784] 벡터로 펼칩니다.
            nn.Flatten(),

            # 첫 번째 완전연결층입니다. 784개 픽셀값을 128차원 특징으로 변환합니다.
            nn.Linear(28 * 28, M_DIS),

            # LeakyReLU는 감별자에서도 음수 영역의 기울기를 유지하여 학습 안정성을 높입니다.
            nn.LeakyReLU(0.01),

            # 출력층입니다. 128차원 특징을 하나의 판별 점수로 변환합니다.
            nn.Linear(M_DIS, 1),

            # Sigmoid는 출력값을 0~1 사이의 확률값으로 변환합니다.
            nn.Sigmoid()
        )

    # forward 메서드는 이미지가 감별자를 통과하는 계산 흐름을 정의합니다.
    def forward(self, img):
        # 입력 이미지를 감별자 모델에 통과시켜 실제 이미지일 확률을 계산합니다.
        validity = self.model(img)

        # 실제 이미지일 확률을 반환합니다.
        return validity

# 감별자 모델 객체를 생성합니다.
discriminator = Discriminator()

# 감별자 모델을 GPU 또는 CPU 장치로 이동합니다.
discriminator = discriminator.to(device)

# 감별자 구조를 출력하여 계층 구성을 확인합니다.
print(discriminator)

## 6. 손실 함수와 최적화 알고리즘 설정

GAN은 생성자와 감별자를 따로 업데이트합니다.  
따라서 생성자용 최적화 알고리즘과 감별자용 최적화 알고리즘을 각각 생성합니다.

In [ ]:
# BCELoss는 이진 분류 문제에서 사용하는 Binary Cross Entropy 손실 함수입니다.
criterion = nn.BCELoss()

# 생성자 파라미터를 업데이트할 Adam 최적화 알고리즘을 생성합니다.
optimizer_G = optim.Adam(
    # 생성자 내부의 학습 가능한 모든 파라미터를 최적화 대상으로 지정합니다.
    generator.parameters(),

    # 학습률을 지정합니다.
    lr=M_LR,

    # Adam의 모멘텀 관련 계수를 지정합니다.
    betas=(M_BETA1, 0.999)
)

# 감별자 파라미터를 업데이트할 Adam 최적화 알고리즘을 생성합니다.
optimizer_D = optim.Adam(
    # 감별자 내부의 학습 가능한 모든 파라미터를 최적화 대상으로 지정합니다.
    discriminator.parameters(),

    # 감별자 학습률을 지정합니다.
    lr=M_LR,

    # Adam의 모멘텀 관련 계수를 지정합니다.
    betas=(M_BETA1, 0.999)
)

# 손실 함수와 최적화 알고리즘 설정 완료 메시지를 출력합니다.
print("손실 함수와 최적화 알고리즘 설정 완료")

## 7. 생성 이미지 저장 함수 작성

학습 중 일정 간격마다 생성자가 만든 이미지를 4×4 격자로 저장합니다.  
이를 통해 학습이 진행되면서 생성 이미지가 어떻게 변화하는지 확인할 수 있습니다.

In [ ]:
# sample 함수는 현재 생성자가 만든 이미지를 파일로 저장하는 함수입니다.
def sample(epoch):
    # 이미지를 생성할 때는 평가 모드로 전환하여 일관된 결과를 얻습니다.
    generator.eval()

    # 이미지 저장 단계에서는 가중치를 업데이트하지 않으므로 기울기 계산을 비활성화합니다.
    with torch.no_grad():
        # 4행 4열로 총 16개의 이미지를 생성합니다.
        row = 4
        col = 4

        # 표준정규분포에서 16개의 노이즈 벡터를 생성합니다.
        noise = torch.randn(row * col, M_NOISE, device=device)

        # 생성자에 노이즈를 입력하여 가짜 이미지를 생성합니다.
        fake_images = generator(noise)

        # 생성 이미지의 값 범위는 -1~1이므로 시각화를 위해 0~1 범위로 변환합니다.
        fake_images = (fake_images + 1) / 2

        # GPU 텐서일 수 있으므로 CPU로 옮긴 뒤 numpy 배열로 변환합니다.
        fake_images = fake_images.cpu().numpy()

    # 4행 4열 이미지를 담을 그림판을 생성합니다.
    fig, axes = plt.subplots(row, col, figsize=(5, 5))

    # 출력할 이미지 번호를 관리하는 변수입니다.
    index = 0

    # 행 방향으로 반복합니다.
    for i in range(row):
        # 열 방향으로 반복합니다.
        for j in range(col):
            # 흑백 이미지 한 장을 해당 위치에 출력합니다.
            axes[i, j].imshow(fake_images[index, 0], cmap="gray")

            # 이미지 주변의 x축과 y축 눈금을 숨깁니다.
            axes[i, j].axis("off")

            # 다음 이미지를 출력하기 위해 인덱스를 1 증가시킵니다.
            index += 1

    # 이미지 사이 여백을 줄여 격자가 깔끔하게 보이도록 합니다.
    plt.tight_layout()

    # 현재 epoch 번호를 파일명에 포함하여 저장 경로를 만듭니다.
    save_path = os.path.join(M_FOLDER, f"epoch_{epoch:05d}.png")

    # 생성된 이미지 격자를 png 파일로 저장합니다.
    plt.savefig(save_path)

    # 화면에 바로 표시하지 않고 메모리를 정리하기 위해 그림 객체를 닫습니다.
    plt.close(fig)

    # 이미지 생성 후 다시 학습 모드로 전환합니다.
    generator.train()

## 8. GAN 학습 함수 작성

GAN 학습은 한 반복에서 크게 두 단계로 진행됩니다.

첫째, 감별자는 실제 이미지를 1로, 생성 이미지를 0으로 맞히도록 학습합니다.  
둘째, 생성자는 감별자가 생성 이미지를 1로 판단하도록 학습합니다.

In [ ]:
# train_gan 함수는 전체 GAN 학습 과정을 수행합니다.
def train_gan():
    # 학습 시작 시간을 기록합니다.
    begin = time()

    # 학습 시작 메시지를 출력합니다.
    print("GAN 학습 시작")

    # DataLoader를 반복해서 사용할 수 있도록 iterator로 변환합니다.
    data_iter = iter(train_loader)

    # 0부터 M_EPOCH까지 반복하여 학습을 진행합니다.
    for epoch in range(M_EPOCH + 1):
        # 현재 반복에서 사용할 실제 이미지 배치를 가져옵니다.
        try:
            # real_images는 실제 MNIST 이미지이고, _는 사용하지 않는 숫자 라벨입니다.
            real_images, _ = next(data_iter)

        # 데이터셋을 한 바퀴 모두 사용하면 StopIteration 예외가 발생합니다.
        except StopIteration:
            # DataLoader iterator를 다시 만들어 처음부터 데이터를 가져오게 합니다.
            data_iter = iter(train_loader)

            # 새 iterator에서 첫 번째 배치를 가져옵니다.
            real_images, _ = next(data_iter)

        # 실제 이미지를 학습 장치로 이동합니다.
        real_images = real_images.to(device)

        # 현재 배치의 이미지 개수를 구합니다.
        batch_size = real_images.size(0)

        # 실제 이미지에 대한 정답 라벨을 1로 생성합니다.
        real_labels = torch.ones(batch_size, 1, device=device)

        # 생성 이미지에 대한 정답 라벨을 0으로 생성합니다.
        fake_labels = torch.zeros(batch_size, 1, device=device)

        # =============================
        # 1단계: 감별자 학습
        # =============================

        # 감별자 최적화기에 저장된 이전 기울기를 모두 0으로 초기화합니다.
        optimizer_D.zero_grad()

        # 감별자에 실제 이미지를 입력하여 실제 이미지일 확률을 계산합니다.
        real_outputs = discriminator(real_images)

        # 실제 이미지를 1로 맞히도록 감별자 손실을 계산합니다.
        d_loss_real = criterion(real_outputs, real_labels)

        # 생성자 입력으로 사용할 무작위 노이즈를 생성합니다.
        noise = torch.randn(batch_size, M_NOISE, device=device)

        # 생성자가 노이즈를 바탕으로 생성 이미지를 만듭니다.
        fake_images = generator(noise)

        # 생성 이미지를 감별자에 입력하여 실제 이미지일 확률을 계산합니다.
        # detach는 감별자 학습 중 생성자의 가중치가 함께 업데이트되지 않게 막습니다.
        fake_outputs = discriminator(fake_images.detach())

        # 생성 이미지를 0으로 맞히도록 감별자 손실을 계산합니다.
        d_loss_fake = criterion(fake_outputs, fake_labels)

        # 실제 이미지 손실과 생성 이미지 손실을 평균내어 감별자 전체 손실을 만듭니다.
        d_loss = 0.5 * (d_loss_real + d_loss_fake)

        # 감별자 손실에 대해 역전파를 수행하여 기울기를 계산합니다.
        d_loss.backward()

        # 계산된 기울기를 사용하여 감별자의 가중치를 업데이트합니다.
        optimizer_D.step()

        # 실제 이미지를 실제 이미지로 맞힌 비율을 계산합니다.
        real_accuracy = (real_outputs >= 0.5).float().mean().item()

        # 생성 이미지를 생성 이미지로 맞힌 비율을 계산합니다.
        fake_accuracy = (fake_outputs < 0.5).float().mean().item()

        # 감별자 정확도는 실제 이미지 정답률과 생성 이미지 정답률의 평균입니다.
        d_accuracy = 0.5 * (real_accuracy + fake_accuracy)

        # =============================
        # 2단계: 생성자 학습
        # =============================

        # 생성자 최적화기에 저장된 이전 기울기를 모두 0으로 초기화합니다.
        optimizer_G.zero_grad()

        # 생성자 학습에 사용할 새로운 무작위 노이즈를 생성합니다.
        noise = torch.randn(batch_size, M_NOISE, device=device)

        # 생성자가 노이즈를 바탕으로 새 생성 이미지를 만듭니다.
        generated_images = generator(noise)

        # 생성 이미지를 감별자에 넣어 실제 이미지일 확률을 계산합니다.
        outputs = discriminator(generated_images)

        # 생성자는 감별자가 생성 이미지를 실제 이미지라고 판단하게 만들어야 하므로 목표 라벨을 1로 둡니다.
        g_loss = criterion(outputs, real_labels)

        # 생성자 손실에 대해 역전파를 수행하여 생성자 가중치의 기울기를 계산합니다.
        g_loss.backward()

        # 계산된 기울기를 사용하여 생성자의 가중치를 업데이트합니다.
        optimizer_G.step()

        # 100회마다 손실과 정확도를 출력하여 학습 상태를 확인합니다.
        if epoch % 100 == 0:
            # 현재 epoch의 감별자 손실, 생성자 손실, 감별자 정확도를 출력합니다.
            print(
                f"Epoch [{epoch:05d}/{M_EPOCH}] "
                f"D Loss: {d_loss.item():.4f} "
                f"G Loss: {g_loss.item():.4f} "
                f"D Acc: {d_accuracy:.4f}"
            )

        # 500회마다 생성 이미지를 output 폴더에 저장합니다.
        if epoch % 500 == 0:
            # 현재 생성자 상태로 샘플 이미지를 저장합니다.
            sample(epoch)

    # 학습 종료 시간을 기록합니다.
    end = time()

    # 전체 학습 시간을 초 단위로 출력합니다.
    print(f"학습 완료: {end - begin:.2f}초")

## 9. GAN 학습 실행

아래 셀을 실행하면 GAN 학습이 시작됩니다.  
CPU 환경에서는 시간이 오래 걸릴 수 있으므로, 먼저 `M_EPOCH` 값을 줄여 테스트한 뒤 최종 학습에서 늘리는 방식이 좋습니다.

In [ ]:
# 생성자를 학습 모드로 설정합니다.
generator.train()

# 감별자를 학습 모드로 설정합니다.
discriminator.train()

# 전체 GAN 학습 함수를 실행합니다.
train_gan()

## 10. 저장된 생성 이미지 확인

학습 중 `output` 폴더에 저장된 생성 이미지 중 가장 마지막 이미지를 출력합니다.

In [ ]:
# output 폴더에 저장된 png 이미지 파일 목록을 정렬하여 가져옵니다.
image_files = sorted(glob.glob(os.path.join(M_FOLDER, "*.png")))

# 저장된 이미지 개수를 출력합니다.
print("저장된 이미지 개수:", len(image_files))

# 저장된 이미지가 하나 이상 있는지 확인합니다.
if len(image_files) > 0:
    # 가장 마지막에 저장된 이미지 파일 경로를 가져옵니다.
    last_image_path = image_files[-1]

    # 마지막 이미지 파일 경로를 출력합니다.
    print("마지막 생성 이미지:", last_image_path)

    # 이미지 파일을 읽어 numpy 배열로 가져옵니다.
    img = plt.imread(last_image_path)

    # 이미지를 화면에 출력합니다.
    plt.imshow(img)

    # 축 눈금을 숨깁니다.
    plt.axis("off")

    # 그래프 제목을 표시합니다.
    plt.title("Last generated sample")

    # 이미지를 화면에 보여줍니다.
    plt.show()
else:
    # 저장된 이미지가 없을 때 안내 문구를 출력합니다.
    print("저장된 이미지가 없습니다. 먼저 train_gan()을 실행하세요.")

## 11. 모델 가중치 저장

학습된 생성자와 감별자의 가중치를 `.pth` 파일로 저장합니다.  
저장된 가중치는 나중에 다시 불러와 이미지 생성이나 추가 학습에 사용할 수 있습니다.

In [ ]:
# 생성자 모델의 학습된 가중치만 저장합니다.
torch.save(generator.state_dict(), "generator_mnist_gan.pth")

# 감별자 모델의 학습된 가중치만 저장합니다.
torch.save(discriminator.state_dict(), "discriminator_mnist_gan.pth")

# 저장이 완료되었음을 출력합니다.
print("모델 저장 완료: generator_mnist_gan.pth, discriminator_mnist_gan.pth")

## 12. 저장된 생성자 불러와 이미지 생성하기

아래 코드는 저장된 생성자 가중치를 다시 불러온 뒤 새로운 노이즈를 입력하여 이미지를 생성하는 예제입니다.

In [ ]:
# 새 생성자 객체를 만듭니다.
loaded_generator = Generator()

# 새 생성자 객체를 GPU 또는 CPU 장치로 이동합니다.
loaded_generator = loaded_generator.to(device)

# 저장된 생성자 가중치를 현재 장치 기준으로 불러옵니다.
loaded_generator.load_state_dict(torch.load("generator_mnist_gan.pth", map_location=device))

# 불러온 생성자를 평가 모드로 전환합니다.
loaded_generator.eval()

# 가중치 업데이트가 필요 없으므로 기울기 계산을 끕니다.
with torch.no_grad():
    # 새로운 무작위 노이즈 1개를 생성합니다.
    test_noise = torch.randn(1, M_NOISE, device=device)

    # 불러온 생성자에 노이즈를 입력하여 이미지 1장을 생성합니다.
    test_image = loaded_generator(test_noise)

    # 생성 이미지 값을 -1~1 범위에서 0~1 범위로 변환합니다.
    test_image = (test_image + 1) / 2

    # 화면 출력이 가능하도록 CPU numpy 배열로 변환합니다.
    test_image = test_image.squeeze().cpu().numpy()

# 생성 이미지를 흑백으로 출력합니다.
plt.imshow(test_image, cmap="gray")

# 축 눈금을 숨깁니다.
plt.axis("off")

# 이미지 제목을 표시합니다.
plt.title("Generated image from loaded generator")

# 이미지를 화면에 보여줍니다.
plt.show()

## 13. 핵심 정리

이 노트북의 전체 흐름은 다음과 같습니다.

1. MNIST 이미지를 불러오고 `[-1, 1]` 범위로 정규화합니다.
2. 생성자는 100차원 노이즈를 입력받아 28×28 이미지를 생성합니다.
3. 감별자는 입력 이미지가 실제 이미지인지 생성 이미지인지 판별합니다.
4. 감별자와 생성자를 번갈아 학습시키며 생성 이미지 품질을 개선합니다.
5. 학습 중 생성 이미지를 저장하고, 학습된 모델 가중치를 `.pth` 파일로 저장합니다.